## Imports

In [50]:
import pandas as pd
import joblib

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

## HS Model

### Load and Prep Data

In [51]:
data = pd.read_excel('RecruitmentPrediction.xlsx', sheet_name='247Data')

data = data[[
    '247Top', 'Position', 'Utah', 'Distance', 'Height', 'Weight',
    'Score', 'LDS', 'Alumni', 'Poly', 'BYU'
]]

categorical_cols = ['247Top', 'Position', 'Utah', 'LDS', 'Alumni', 'Poly']

for col in categorical_cols:
    data[col] = data[col].astype('category')

data['BYU'] = data['BYU'].map({'N': 0, 'Y': 1})

X = data.drop(columns=['BYU'])
y = data['BYU']

### Preprocessing

In [52]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

### Pipeline

In [53]:
pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=123)),
    ('model', HistGradientBoostingClassifier(random_state=123))
])

### CV Setup

In [54]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

param_grid = {
    'model__max_depth': [4, 6, None],
    'model__learning_rate': [0.03, 0.05],
    'model__max_iter': [200, 300]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

### Fit

In [55]:
grid_search.fit(X, y)

print("Best params:", grid_search.best_params_)
print("Best ROC AUC:", grid_search.best_score_)

Best params: {'model__learning_rate': 0.05, 'model__max_depth': 6, 'model__max_iter': 300}
Best ROC AUC: 0.8491082603254068


### Accuracy Test

In [56]:
best_model = grid_search.best_estimator_

probs = best_model.predict_proba(X)[:, 1]
preds = (probs >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y, preds))

acc_scores = cross_val_score(
    best_model, X, y,
    cv=5,
    scoring='accuracy'
)

print("CV Accuracy:", acc_scores.mean())

Accuracy: 1.0
CV Accuracy: 0.767361111111111


### Save

In [57]:
joblib.dump(grid_search.best_estimator_, "hs_model.pkl")

['hs_model.pkl']

## Transfer/JC Model

### Load and Prep Data

In [58]:
data = pd.read_excel('RecruitmentPrediction.xlsx', sheet_name='Transfers')

data = data[[
    'Years', '247Top', 'Position', 'Distance', 'Conf', 'Height', 'Weight',
    'Score', 'LDS', 'Alumni', 'Prev', 'Poly', 'BYU'
]]

categorical_cols = ['247Top', 'Position', 'Conf', 'LDS', 'Alumni', 'Prev', 'Poly']

for col in categorical_cols:
    data[col] = data[col].astype('category')

data['BYU'] = data['BYU'].map({'N': 0, 'Y': 1})

X = data.drop(columns='BYU')
y = data['BYU']

### Preprocessing

In [59]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

### Pipeline

In [60]:
pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=123)),
    ('model', HistGradientBoostingClassifier(random_state=123))
])

### CV Setup

In [61]:
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

### Fit

In [62]:
grid_search.fit(X, y)

print("Best params:", grid_search.best_params_)
print("Best ROC AUC:", grid_search.best_score_)

Best params: {'model__learning_rate': 0.03, 'model__max_depth': 4, 'model__max_iter': 300}
Best ROC AUC: 0.7798484848484849


### Accuracy Test

In [63]:
best_model = grid_search.best_estimator_

probs = best_model.predict_proba(X)[:, 1]
preds = (probs >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y, preds))

acc_scores = cross_val_score(
    best_model, X, y,
    cv=5,
    scoring='accuracy'
)

print("CV Accuracy:", acc_scores.mean())

Accuracy: 0.9358974358974359
CV Accuracy: 0.7175


### Save

In [64]:
joblib.dump(grid_search.best_estimator_, "transfer_jc_model.pkl")

['transfer_jc_model.pkl']